In [6]:
%pip install sentence-transformers

Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer

W1111 13:08:51.945000 11220 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


In [2]:
books_df = pd.read_csv('BooksDatasetClean.csv')

# all-MiniLM-L6-v2 is This is a sentence-transformers model
# It maps sentences & paragraphs to a 384 dimensional dense vector space and can be used for tasks like clustering or semantic search
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [3]:
# Fill missing titles or descriptions with an empty string
books_df['Title'] = books_df['Title'].fillna('')
books_df['Description'] = books_df['Description'].fillna('')
books_df['Category'] = books_df['Category'].fillna('')

text_to_encode = (books_df['Title'] + " " + books_df['Description'] + " " + books_df['Category']).to_list()

# Filter out any rows that ended up completely empty
text_to_encode = [str(text) for text in text_to_encode if text.strip() != ""]
book_embeddings = model.encode(text_to_encode, show_progress_bar=True)

Batches:   0%|          | 0/3221 [00:00<?, ?it/s]

In [5]:
# save the embeddings to a file for later use
np.save('book_embeddings.npy', book_embeddings)

In [ ]:
def vibe_search_book(query, top_k=10):
    query_embedding = model.encode([query])
    similarities = np.dot(book_embeddings, query_embedding.T).flatten()
    top_indices = similarities.argsort()[-top_k:][::-1]
    return books_df.iloc[top_indices]

vibe_search_book("military")

,Title,Authors,Description,Category,Publisher,Price Starting With ($),Publish Date (Month),Publish Date (Year)
27351,Soviet Army,"By Seaton, Albert and Seaton, Joan",,,Dutton Adult,5.33,March,1987
49283,The Marines,"By Simmons, Edward Howard -Editor -etal.",,,Marine Corps Heritage Foundation,9.53,October,1998
14019,National Defense,"By Fallows, James M.",,,Vintage,5.29,March,1982
16582,Modern U.S. Army,"By Jordan, John",,,Smithmark Pub,5.29,March,1992
9139,"The Army means business,",By Herbert Corey,,,The Bobbs-Merrill Company,27.69,January,1942
35869,Army Trivia,"By Burke, Edward J.",,,Quinlan Pr,6.58,August,1985
76523,U.S. Special Operations Forces,By Smithsonian Institution National Air and Sp...,,,Universe,9.07,January,1995
19844,Marching Toward the 21st Century: Military Man...,"By Eitelberg, Mark J. and Mehay, Stephen L. (EDT)",This book will appeal to persons interested in...,"Political Science , International Relations ,...",Praeger,4.99,June,1994
17832,Military Technology/#07575,By,,,Bookthrift Co,7.09,February,1987
28196,Military Power And The Advance Of Technology: ...,"By Deitchman, Seymour J.",,"Political Science , Security (National & Inte...",Westview Press,5.99,September,1983


In [6]:
from sentence_transformers.util import semantic_search

def vibe_search_kitabu(query, top_k=10):
    query_embedding = model.encode(query)
    # cosine similarity
    hits = semantic_search(query_embedding, np.load('book_embeddings.npy'), top_k=top_k)
    for hit in hits[0]:
        idx = hit['corpus_id']
        print(f"Match: {books_df.iloc[idx]['Title']} (Score: {hit['score']:.2f})")

vibe_search_kitabu("women and golf")

Match: The Wisdom of Women's Golf (Score: 0.83)
Match: Golf For Women: Easy-to-follow Instruction from Pro Golf's Leading Tournament Winner (Score: 0.75)
Match: The Golf Swing (Score: 0.69)
Match: Golfoholics: The Front Nine (Score: 0.68)
Match: A Woman's Guide to Better Golf (Score: 0.67)
Match: The Love of Golf (Score: 0.66)
Match: Golf Handbook for Women: The Complete Guide to Improving Your Game (Score: 0.66)
Match: PLAY BETTER GOLF (Score: 0.65)
Match: Play Better Golf (Golf Clinic) (Score: 0.64)
Match: Visual Golf (Score: 0.64)


In [7]:
len(books_df)

103063